### Analysis

In [89]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px


datafile = 'Machinek-PRF-trunc'

inpath = f'../data/post_data/{datafile}/preprocess_Machinek-PRF.npz'
loaded_data = np.load(inpath, allow_pickle=True)
trans_time = loaded_data["trans_time"]

inpath4 = f'../data/post_data/{datafile}/time_Machinek-PRF.npz'
loaded_data4 = np.load(inpath4, allow_pickle=True)
trj_id = loaded_data4['trj_id']


In [90]:
rxnTimes = trans_time[trj_id]
ids = np.arange(rxnTimes.shape[0])

sortTimes = np.sort(rxnTimes)[::-1]
sortIDs = ids[np.argsort(rxnTimes)[::-1]]


0.31602669386911414

In [127]:
fig = px.scatter(x=ids, y=sortTimes, hover_data=[sortIDs], log_y=True)
# fig = px.scatter(x=ids, y=sortTimes, hover_data=[sortIDs])

# plot a horizontal line for the average time
fig.add_hline(y=np.mean(rxnTimes), line_dash="dot", line_color="red")
fig.add_hline(y=1e-2, line_dash="dot", line_color="green")


fig.show()

In [ ]:
np.mean(sortTimes), np.max(sortTimes), np.min(sortTimes)

### Statistics

In [1]:
import numpy as np
import re
import pickle
import gzip
import pandas as pd
import sys
sys.path.append('/Users/chenwei/Desktop/Github/ViDa') 


import imp, vida.data_processing.strandReorder
imp.reload(vida.data_processing.strandReorder)
from vida.data_processing.strandReorder import *

# import imp, vida.data_processing.utils
# imp.reload(vida.data_processing.utils)
# from vida.data_processing.utils import *

import imp, vida.adjmat.dp2adj
imp.reload(vida.adjmat.dp2adj)
from vida.adjmat.dp2adj import *

import imp, vida.data_processing.comp_time
imp.reload(vida.data_processing.comp_time)
from vida.data_processing.comp_time import *


/var/folders/z9/3zs_cg3x09n1q9fmthyx2x600000gn/T/ipykernel_2876/3343276200.py:10: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  import imp, vida.data_processing.strandReorder


In [2]:
datafile='Machinek-PRF-trunc'

ref_strands = 'CCCTCCACATTCAACCTCAAACTCACC+TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA+GGTGAGTTTGAGGTTGAATGTGGA'
strand_sub = 'CCCTCCACATTCAACCTCAAACTCACC'  # substrate_perf_seq
strand_incb = 'TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA'  # incumbent_perf_seq
strand_inv = 'GGTGAGTTTGAGGTTGAATGTGGA'  # invader_perf_seq

In [ ]:
#            inv     (correct)         (toehold)      sub                  (correct)             incb
# GGTGAGTTTGAGGTTGA   ATGTGGA         CCC TCCACAT  TCAACCTCAAACTCACC      TGGTGTTTGTGGGTGT   GGTGAGTTTGAGGTTGA



#           incb                           (toehold)      sub                         inv
# TGGTGTTTGTGGGTGT GGTGAGTTTGAGGTTGA     CCC TCCACAT  TCAACCTCAAACTCACC      GGTGAGTTTGAGGTTGAATGTGGA           



### mismatch inv + sub    ####
#    ATGTG-GAGG-G      C-CCTC-CACAT



# initial
#                     incb                         sub                      inv
#   ................(((((((((((((((((+..........))))))))))))))))) ........................


# final
#            incb                               sub                      inv
#  ................................. ...((((((((((((((((((((((((+))))))))))))))))))))))))



In [3]:
inpath2 = f'../data/post_data/{datafile}/Machinek-PRF.pkl.gz'
with gzip.open(inpath2, 'rb') as file:
    load_data_seq = pickle.load(file)

ref_name = load_data_seq["ref_name"]
ref_name_list = load_data_seq["ref_name_list"]
strand_list = load_data_seq["strand_list"]
trajs_seqs = load_data_seq["trajs_seqs"]
trajs_states = load_data_seq['trajs_states']
trajs_times = load_data_seq['trajs_times']
trajs_energies = load_data_seq['trajs_energies']
trajs_shortnames = load_data_seq['trajs_shortnames']
trajs_incbinvpairs = load_data_seq['trajs_incbinvpairs']

In [4]:
rxn_times = []
for i in range(len(trajs_times)):
    rxn_times.append(trajs_times[i][-1])
    
rxn_times = np.array(rxn_times)

thres = np.float64(1e-2)

print('Max reaction time:', np.max(rxn_times), 'seconds')
print('Min reaction time:', np.min(rxn_times), 'seconds')
print('Average reaction time:', np.mean(rxn_times), 'seconds')
print('Median reaction time:', np.median(rxn_times), 'seconds')
print(f'Number of reaction slower than {thres}:', np.where(rxn_times > thres)[0].shape[0])
print(f'Number of reaction faster than {thres}:', np.where(rxn_times < thres)[0].shape[0])


Max reaction time: 2.4719820372770687 seconds
Min reaction time: 0.0006404025694753318 seconds
Average reaction time: 0.31602669386911414 seconds
Median reaction time: 0.14070118945995877 seconds
Number of reaction slower than 0.01: 276
Number of reaction faster than 0.01: 124


In [5]:
thres = np.float64(1e-3)
thres

0.001

In [6]:
mismatchIntraj = np.zeros(len(trajs_shortnames), dtype=int)

for i, shortnames in enumerate(trajs_shortnames):
    
    for j, name in enumerate(shortnames):
        if name == 'inv+sub+incb':
            inv, sub, _ = trajs_states[i][j].split('+')
            
            # identify the mismatched strand
            if '))))' in sub[0:5] and '((((' in inv:
                # if sub[0] == ')' or sub[1] == ')' or sub[2] == ')':
                mismatchIntraj[i] = 1 
                
                
                # if i in [275, 117, 287, 270, 259]:
                #     print(i)
                #     print(trajs_states[i][j])
                #     print(sub, inv)
                # break
            
        if name == 'incb+sub+inv': 
            _, sub, inv = trajs_states[i][j].split('+') 
            
            # identify the mismatched strand
            if '((((' in sub[0:5] and '))))' in inv:
                mismatchIntraj[i] = 1 
                break   
    

In [7]:
traj_mismatch = np.where(mismatchIntraj == 1)[0]
traj_mismatch.shape[0], traj_mismatch

(265,
 array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
         15,  17,  19,  20,  21,  22,  25,  26,  27,  28,  29,  31,  32,
         33,  34,  35,  36,  37,  39,  40,  41,  42,  43,  44,  45,  48,
         49,  50,  51,  53,  54,  56,  58,  59,  60,  65,  66,  69,  71,
         72,  74,  75,  76,  77,  78,  80,  81,  82,  86,  88,  89,  91,
         92,  93,  94,  96, 100, 102, 103, 106, 107, 108, 111, 113, 115,
        116, 117, 118, 119, 120, 122, 123, 125, 129, 133, 135, 136, 138,
        141, 143, 146, 147, 149, 150, 151, 153, 154, 157, 158, 159, 160,
        162, 165, 166, 167, 168, 169, 171, 172, 173, 174, 176, 178, 180,
        181, 182, 185, 186, 187, 188, 189, 190, 191, 192, 194, 196, 197,
        198, 199, 200, 202, 203, 204, 205, 209, 212, 213, 214, 215, 218,
        219, 220, 222, 223, 224, 227, 228, 230, 231, 233, 234, 235, 236,
        238, 239, 241, 242, 244, 245, 248, 250, 251, 253, 258, 259, 260,
        262, 263, 265, 266, 268, 269, 270, 27

In [8]:
traj_no_mismatch = np.where(mismatchIntraj == 0)[0]

traj_no_mismatch.shape[0], traj_no_mismatch

(135,
 array([  0,  14,  16,  18,  23,  24,  30,  38,  46,  47,  52,  55,  57,
         61,  62,  63,  64,  67,  68,  70,  73,  79,  83,  84,  85,  87,
         90,  95,  97,  98,  99, 101, 104, 105, 109, 110, 112, 114, 121,
        124, 126, 127, 128, 130, 131, 132, 134, 137, 139, 140, 142, 144,
        145, 148, 152, 155, 156, 161, 163, 164, 170, 175, 177, 179, 183,
        184, 193, 195, 201, 206, 207, 208, 210, 211, 216, 217, 221, 225,
        226, 229, 232, 237, 240, 243, 246, 247, 249, 252, 254, 255, 256,
        257, 261, 264, 267, 271, 273, 274, 280, 282, 284, 286, 294, 302,
        304, 306, 308, 313, 314, 316, 317, 325, 326, 331, 333, 334, 335,
        338, 348, 350, 357, 358, 359, 361, 363, 364, 365, 373, 374, 376,
        379, 380, 381, 382, 397]))

In [9]:
np.sort(rxn_times[traj_mismatch])[:10], traj_mismatch[np.argsort(rxn_times[traj_mismatch])[:5]]


(array([0.00382539, 0.00430639, 0.00565632, 0.00739233, 0.00803268,
        0.01060342, 0.01447687, 0.01551184, 0.0159011 , 0.01600097]),
 array([275, 117, 287, 270, 259]))

In [63]:
hairpinInmatch = np.zeros(len(traj_no_mismatch), dtype=int)

for i, idx in enumerate(traj_no_mismatch):
    for j, name in enumerate(trajs_shortnames[idx]):
        if name == 'inv+sub+incb':
            inv, _, _ = trajs_states[idx][j].split('+')
            
            if '(((' in inv[6:17] and ')))' in inv[6:17]:
                hairpinInmatch[i] = 1
                break
        
        if name == 'incb+sub+inv':
            _, _, inv = trajs_states[idx][j].split('+')
            
            if '(((' in inv[6:17] and ')))' in inv[6:17]:
                hairpinInmatch[i] = 1
                break
        
        if name == 'incb+sub inv':
            _, _, inv = re.split(r'\s|\+', trajs_states[idx][j]) 
            
            if '(((' in inv[6:17] and ')))' in inv[6:17]:
                hairpinInmatch[i] = 1
                break
    

In [73]:
np.where(hairpinInmatch==1)[0].shape[0], np.where(hairpinInmatch==1)[0], 118/135

(128,
 array([  0,   1,   2,   3,   4,   5,   6,   8,   9,  10,  11,  12,  13,
         15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
         28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  40,  41,
         42,  43,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
         56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,
         69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,
         82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,
         96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 108, 109,
        110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122,
        123, 124, 125, 127, 128, 129, 130, 131, 132, 133, 134]),
 0.8740740740740741)

In [65]:
traj_no_mismatch[np.where(hairpinInmatch==1)[0]]

array([  0,  14,  16,  18,  23,  24,  30,  46,  47,  52,  55,  57,  61,
        63,  64,  67,  68,  70,  73,  83,  84,  85,  87,  95,  97,  98,
        99, 101, 104, 105, 109, 110, 112, 114, 121, 126, 127, 128, 132,
       134, 137, 139, 140, 142, 144, 145, 148, 152, 155, 156, 163, 164,
       170, 175, 177, 179, 183, 184, 195, 201, 206, 207, 208, 210, 211,
       216, 217, 221, 225, 226, 232, 240, 243, 246, 247, 252, 254, 255,
       256, 257, 261, 267, 273, 274, 280, 282, 284, 286, 294, 302, 304,
       306, 308, 314, 316, 317, 325, 331, 333, 334, 335, 338, 348, 350,
       357, 358, 359, 361, 363, 364, 373, 374, 376, 379, 380, 381, 382,
       397])

In [66]:
rxn_times[256]

0.006083609291018341

In [68]:
hairpinInmismatch = np.zeros(len(traj_mismatch), dtype=int)

for i, idx in enumerate(traj_mismatch):
    for j, name in enumerate(trajs_shortnames[idx]):
        if name == 'inv+sub+incb':
            inv, _, _ = trajs_states[idx][j].split('+')
            
            if '(((' in inv[6:17] and ')))' in inv[6:17]:
                hairpinInmismatch[i] = 1
                break
        
        if name == 'incb+sub+inv':
            _, _, inv = trajs_states[idx][j].split('+')
            
            if '(((' in inv[6:17] and ')))' in inv[6:17]:
                hairpinInmismatch[i] = 1
                break
        
        if name == 'incb+sub inv':
                _, _, inv = re.split(r'\s|\+', trajs_states[idx][j]) 
                
                if '(((' in inv[6:17] and ')))' in inv[6:17]:
                    hairpinInmismatch[i] = 1
                    break

In [69]:
np.where(hairpinInmismatch==1)[0].shape[0], np.where(hairpinInmismatch==1)[0], len(traj_mismatch)

(265,
 array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
         13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
         39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
         52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
         65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
         78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
         91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
        104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
        117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
        130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
        143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
        156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
        169, 170, 171, 172, 173, 174, 175, 17

In [70]:
traj_mismatch[np.where(hairpinInmismatch==0)[0]], rxn_times[136]

(array([], dtype=int64), 0.05134682087266487)

In [81]:
trajs_states[375][-500:]

['...............(((((((((+...)))))))))(((((((((((((((+................)))))))))))))))..',
 '...............(((((((((+...)))))))))..(((((((((((((+................)))))))))))))....',
 '...............(((((((((+...)))))))))..(((((((((((((+................)))))))))))))....',
 '..........((((((((((((((+...))))))))))))))((((((((((+................)))))))))).......',
 '..........((((((((((((((+...))))))))))))))((((((((((+................)))))))))).......',
 '...........((((((((((((.+....)))))))))))).((((((((((+................)))))))))).......',
 '............((((((((((((+...))))))))))))((((((((((((+................)))))))))))).....',
 '............((((((((((((+...))))))))))))((((((((((((+................)))))))))))).....',
 '...............(((((((((+...)))))))))(((((((((((((((+................)))))))))))))))..',
 '..............(((((((((.+....))))))))).(((((((((((((+................)))))))))))))....',
 '...............(((((((((+...)))))))))(((((((((((((((+................)))))))))))))))..',